In [1]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 03, Poisson geometry

Companion notebook to [03_poisson_geometry.md](03_poisson_geometry.md). Three equivalent views of the Poisson bracket on a symplectic manifold (derived, Hamiltonian, Koszul) plus the Jacobi proof reduced to the single condition `[π, π]_SN = 0`.

## Symplectic manifold, `(ω, π, ♭, ♯)` bundle

`SymplecticManifold(ω, bivector=π)` keeps the form, the inverse bivector, the musical maps, and the `MusicalCompatibility` axiom together in one object. On the registry `ω` is a 2-form and `π` is a 2-vector with SN-degree 1, the `Bivector` helper handles that automatically.

In [2]:
from jacopy import Bivector, Forms, Functions
from jacopy.core.registry import PropertyRegistry
from jacopy.library.symplectic import SymplecticManifold

reg = PropertyRegistry()
(omega,) = Forms("ω", degree=2, registry=reg)
pi = Bivector("π", registry=reg)

M = SymplecticManifold(omega, bivector=pi, name="(M, ω, π)")
print(M)
print('flat:', M.flat)
print('sharp:', M.sharp)
print('compat:', M.compatibility.name)

SymplecticManifold(ω, π=π)
flat: ω♭
sharp: π♯
compat: musical compatibility of ω, π


## `PoissonBracket`, three equivalent views

Functions live at SN-shifted degree `−1`; the `Functions` helper takes the `degree=-1` kwarg for this context.

In [3]:
from jacopy.library.poisson import PoissonBracket

f, g, h = Functions("f g h", degree=-1, registry=reg)
poisson = PoissonBracket.from_bivector(pi)
print('bracket:', poisson)

bracket: PoissonBracket(π=π)


### View 1, the derived bracket

`{f, g}_π = [[f, π]_SN, g]_SN`.

In [4]:
poisson.expand(f, g, reg)

[·,·]_SN(f, π)(g)

### View 2, the Hamiltonian vector field

`{f, g}_π = X_f(g)`. On a symplectic manifold this is equivalent to `ι_{X_f} ω + df = 0`; `prove_hamiltonian_equivalence` closes that in five steps using the musical compatibility.

In [5]:
from jacopy.display import chain_to_ascii

print('X_f =', poisson.hamiltonian_vf(f))
print('X_f(g) =', poisson.via_hamiltonian(f, g))

chain = M.prove_hamiltonian_equivalence(f, registry=reg)
print('chain length:', len(chain))
print(chain_to_ascii(chain))

X_f = X_f
X_f(g) = X_f(g)
chain length: 5
[ι_X ω = ω♭(X) [musical compatibility of ω, π]] (axiom) ι_X_f(ω) -> ω♭(X_f)  -- apply axiom: ι_X ω = ω♭(X) [musical compatibility of ω, π]
[X_f = −π♯(df) [X_f]] (axiom) X_f -> -π♯(d(f))  -- apply axiom: X_f = −π♯(df) [X_f]
[D(-x) = -D(x)] (axiom) ω♭(-π♯(d(f))) -> -ω♭(π♯(d(f)))  -- apply axiom: D(-x) = -D(x)
[musical compatibility of ω, π] (axiom) ω♭(π♯(d(f))) -> d(f)  -- apply axiom: musical compatibility of ω, π
[simplify] (-d(f) + d(f)) - 0 -> 0  -- canonical-form pipeline


### View 3, the Koszul three-term formula

On 1-forms `{α, β}_π = L_{π♯(α)} β − L_{π♯(β)} α − d⟨π♯(α), β⟩`. The classical Koszul bracket and the derived bracket are *structurally equal* on this operand type, `prove_koszul_equivalence` records that in a single reflexive step.

In [6]:
alpha, beta = Forms("α β", degree=1, registry=reg)
print('koszul expand:', poisson.koszul_expand(alpha, beta, reg))

chain_k = poisson.prove_koszul_equivalence(alpha, beta, registry=reg)
print('koszul chain length:', len(chain_k))
print('rule:', chain_k.steps[0].rule)

koszul expand: (L_π♯(α)(β) + (-L_π♯(β)(α)) + (-d(⟨π♯(α), β⟩)))
koszul chain length: 1
rule: reflexive


## `[π, π]_SN = 0`, the single condition

The Derived Bracket Theorem reduces Jacobi for `{·, ·}_π` to one condition: `[π, π]_SN = 0`. The three-input reduction chain reaches the obstruction in a single `DerivedBracketTheorem` step; for atomic `π` the obstruction stays opaque, the Jacobi identity closes once the Poisson hypothesis is supplied.

In [7]:
print('obstruction:', poisson.jacobi_obstruction(reg))
print('condition:', poisson.jacobi_condition(reg))

chain_j = poisson.prove_jacobi_reduction(f, g, h, registry=reg)
print('chain length:', len(chain_j))
print('rule:', chain_j.steps[0].rule)
print('reduces to:', chain_j.steps[0].after)

obstruction: [·,·]_SN(π, π)
condition: Poisson Jacobi condition on {·,·}_π: [·,·]_SN(π, π) = 0
chain length: 1
rule: DerivedBracketTheorem
reduces to: [·,·]_SN(π, π)


## Theorem Book, the seeded theorem

The library carries this reduction as a ready `Theorem` entry under `poisson_jacobi`, downstream code wires the result through a single citation.

In [8]:
from jacopy.library import theorem_book

thm = theorem_book.get("poisson_jacobi")
print('statement:', thm.statement)
print('from_axioms:', thm.from_axioms)

statement: {f, g, h}_π cyclic sum = 0 when [π, π]_SN = 0
from_axioms: ('Derived Bracket Theorem', '[π, π]_SN = 0 (Poisson hypothesis)')


## Next step

The Lie algebroid framework applies the same derivation strategy to a bracket living on a vector bundle, [04_lie_algebroid.md](04_lie_algebroid.md).